In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
with open("wikitext_10M.txt", "r", encoding="utf-8") as f:
    faqs = f.read()

print("Characters:", len(faqs))
print(faqs[:500])

Characters: 52825461
= Valkyria Chronicles III = Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs paral


In [3]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.preprocessing.text import Tokenizer

I0000 00:00:1781802310.163823    1490 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781802312.927398    1490 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
tokenizer = Tokenizer()

In [5]:
MAX_WORDS = 5000

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<UNK>"
)

tokenizer.fit_on_texts(faqs.split('\n'))

In [6]:
tokenizer.fit_on_texts([faqs])

In [7]:
input_sequences = []

MAX_LEN = 21

for sentence in faqs.split('\n'):
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1, len(tokenized_sentence)):
        seq = tokenized_sentence[max(0, i + 1 - MAX_LEN):i + 1]
        input_sequences.append(seq)

In [8]:
MAX_LEN = max([len(x) for x in input_sequences])

In [9]:
print(MAX_LEN)

21


In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
padded_input_sequences = pad_sequences(input_sequences, maxlen = MAX_LEN, padding='pre')

In [11]:
X = padded_input_sequences[:,:-1]

In [12]:
y = padded_input_sequences[:,-1]

In [13]:
input_sequences.append(tokenized_sentence[:i+1])

In [14]:
vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary:", vocab_size)
print("X shape:", X.shape)
print("y shape:", y.shape)

Vocabulary: 151475
X shape: (8577915, 20)
y shape: (8577915,)


In [15]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)



checkpoint = ModelCheckpoint(
    "best_next_word.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, BatchNormalization

vocab_size = len(tokenizer.word_index) + 1
sequence_length = X.shape[1]

model = Sequential([
    Input(shape=(X.shape[1],)),
    Embedding(vocab_size, 64),
    LSTM(64),
    Dense(vocab_size, activation="softmax")
])

In [17]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [18]:
history = model.fit(
    X, y,
    epochs=2,
    batch_size=912,
    validation_split=0.2,
    callbacks =
    [early_stop,
    reduce_lr,
    checkpoint]
)

Epoch 1/2
7525/7525 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.1809 - loss: 5.9115
Epoch 1: val_loss improved from None to 5.10881, saving model to best_next_word.keras
7525/7525 ━━━━━━━━━━━━━━━━━━━━ 1660s 220ms/step - accuracy: 0.1979 - loss: 5.4833 - val_accuracy: 0.2180 - val_loss: 5.1088 - learning_rate: 0.0010
Epoch 2/2
7525/7525 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 0.2244 - loss: 4.9716
Epoch 2: val_loss improved from 5.10881 to 4.84545, saving model to best_next_word.keras
7525/7525 ━━━━━━━━━━━━━━━━━━━━ 1637s 217ms/step - accuracy: 0.2273 - loss: 4.9053 - val_accuracy: 0.2289 - val_loss: 4.8454 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 2.


In [19]:
import numpy as np
text = "Artificial Intelligence"

for i in range(24):
    token_text = tokenizer.texts_to_sequences([text])[0]
    padded_token_text = pad_sequences([token_text], maxlen=X.shape[1], padding='pre')

    pred = model.predict(padded_token_text, verbose=0)
    pos = np.argmax(pred, axis=1)[0]

    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == pos:
            output_word = word
            break

    if output_word == "" or output_word == "<UNK>":
        break

    text += " " + output_word

print(text)

Artificial Intelligence
